#### Goal
- To Achive Party Document details from GCOB for now

#### author
- Sowmyashree.parashivamurthy.sudha@rabobank.com

##### flow of logic
- Read Data from GCOB to further transform if needed

##### Version & Changes
  |Developer   |   Date          |   PBI No.     |  Changes done|
  |---|--|---|---|
  |Sowmyashree | 25-July-2025    |   13024196    |  First release| 
  | Mahalakshmi V | 20-Nov-2025 | 14165735 | Made notebook compatible for Historical Load
 


##### Read Files from GDP

In [0]:
import os
app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']
GIC_ReadStorage = os.environ['GDP_SA_STORAGE_NAME']
environment=os.environ['ENV']
radar_datamodel_version_number=3
service_credential = dbutils.secrets.get(scope="connectedsecrets", key = f"appreg-{app_reg_app_id}")
SARADAR = "saradar" + environment

In [0]:
from RadarUtils import *

In [0]:
dbutils.widgets.text("Load_Date", "")
dbutils.widgets.text("RunType", "daily")  # default is daily

Load_Date = dbutils.widgets.get("Load_Date")
RunType = dbutils.widgets.get("RunType").lower()
if Load_Date:
    RunType="historical"
    
if not Load_Date:
    Load_Date = datetime.today().strftime('%Y%m%d')

print(f"Running for Load Date: {Load_Date}")

In [0]:
party_document_object='Party_Documents'

In [0]:
authenticate_storage_account(ReadStorage)
authenticate_storage_account(GIC_ReadStorage)
authenticate_storage_account(SARADAR)

In [0]:
# Derive the date for which data has to be processed from GDP
load_dts = f"EDL_LOAD_DTS={Load_Date}*"
print (load_dts)

In [0]:
# print list of strings for loading spark dfs from GDP
load_df = [
'party_documents'
,'party_case_client_details'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='CaseService',Load_Date=Load_Date)

In [0]:
# List of datasets from GDP
load_df = [
'Legacy2_ClientIdentification',
'Legacy2_case_client_details'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='Legacy2', Dataobject=Object,path_prefix='Legacy2',Load_Date=Load_Date)

In [0]:
df_Party_SystemIdentifier = spark.read.parquet(
    f"abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_SystemIdentifier/3/data/{load_dts}/*.parquet"
)

In [0]:
%sql
Create or replace temporary view Gcob_Documents As
with Document as 
(
select distinct pd.*
,case when c.CaseStatusName is null then 'RelatedParty' else c.CaseStatusName end as StatusName
FROM party_documents AS pd 
left join party_case_client_details c on pd.UniquePartyId = c.UniqueGcobId
WHERE pd.DocumentId IS NOT NULL
)
SELECT DISTINCT
CONCAT('GCOB_', Doc.UniquePartyId) AS LocalSystemIdentifier,
Doc.ClientId,
Doc.DocumentId,
Doc.DocumentExpiryDate,
-- Doc.EntityType,
-- Doc.EntityId,
Doc.DocumentSubTypeName,
Doc.DocumentTypeName,
Doc.DocumentFileName,
Doc.DocumentStoreIdentifier,
Doc.UploadedBy,
Doc.UploadDate,
Doc.ExpiryDate,
Doc.Purpose
FROM Document AS Doc 
WHERE StatusName <> 'Cancelled'

In [0]:
%sql
Create or replace temporary view Party_Documents_Identifier As
select * From Gcob_Documents

In [0]:
df_party_documents_Identifier=spark.table('Party_Documents_Identifier')

In [0]:
df_party_documents_Identifier = add_party_identifier(df_party_documents_Identifier, df_Party_SystemIdentifier)

In [0]:
df_party_documents_Identifier.createOrReplaceTempView('Identifier')

In [0]:
%sql
Create or replace temporary view Final_Party_documents_Identifier As
select * from Identifier where PartyIdentifier is not null

In [0]:
df_Final_Documents=spark.table('Final_Party_documents_Identifier')

In [0]:
if RunType == "historical":
    save_to_saradar_storage_account(df_Final_Documents, party_document_object, radar_datamodel_version_number, environment,Load_Date)
else:
    save_to_saradar_storage_account(df_Final_Documents, party_document_object, radar_datamodel_version_number, environment)